## **Machine Learning Project**

**Import Libraries**

In [248]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
import plotly.express as px

**Load, Explore, Clean, and Deal With Missing Values**

In [249]:
pc_price = pd.read_csv("/Users/_jawaher/Desktop/ML/ML_1/laptopData.csv")
pc_price.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0.0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1.0,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2.0,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,3.0,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,4.0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [250]:
# Clean The Data
pc_price = pc_price.drop(columns='Unnamed: 0')
# There Count Of Duplicated rows
print("Total number droping the duplicated rows:",pc_price.duplicated().sum())
pc_price = pc_price.drop_duplicates()

Total number droping the duplicated rows: 58


In [251]:
# Get the dimensions
print(f"The number of columns ({pc_price.shape[1]}), and the number of rows ({pc_price.shape[0]})")

# Summary about dataset
print(pc_price.describe())
print(pc_price.info())

pc_price.isnull().sum()
# Deal the Missing Values
# Since it is only one value that is null will drop it
pc_price = pc_price.dropna()

The number of columns (11), and the number of rows (1245)
               Price
count    1244.000000
mean    60606.224427
std     37424.636161
min      9270.720000
25%     32655.445200
50%     52693.920000
75%     79813.440000
max    324954.720000
<class 'pandas.core.frame.DataFrame'>
Index: 1245 entries, 0 to 1273
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1244 non-null   object 
 1   TypeName          1244 non-null   object 
 2   Inches            1244 non-null   object 
 3   ScreenResolution  1244 non-null   object 
 4   Cpu               1244 non-null   object 
 5   Ram               1244 non-null   object 
 6   Memory            1244 non-null   object 
 7   Gpu               1244 non-null   object 
 8   OpSys             1244 non-null   object 
 9   Weight            1244 non-null   object 
 10  Price             1244 non-null   float64
dtypes: float64(1), object(10)
memory us

Below are the following operations:

1- **Data Engineering***

2- **Preprocessing (encoding + scaling)**

3-  **PCA (applied + interpreted)**

4- **Visualizing Insights**

In [252]:
heatmap_df = pd.DataFrame()

heatmap_df['Cpu_Encoded'] = pc_price['Cpu'].map(pc_price.groupby('Cpu')['Price'].mean())
heatmap_df['Gpu_Encoded'] = pc_price['Gpu'].map(pc_price.groupby('Gpu')['Price'].mean())
heatmap_df['Price'] = pc_price['Price']

corr_matrix = heatmap_df.corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True, 
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title="Correlation Heatmap: CPU And GPU Impact On Price",
    labels=dict(color="Correlation")
)

fig.show()

Both has similler in impcating the price but if we 

In [253]:
company_price = pc_price.groupby('Company', as_index=False)['Price'].mean()

company_price = company_price.sort_values(by='Price', ascending=False)

fig = px.bar(
    company_price,
    x='Company',
    y='Price',
    title='Average Laptop Price by Company',
    labels={'Price': 'Average Price ($)', 'Company': 'Brand'},
    color='Price',
    color_continuous_scale='Turbo'
)

fig.show()

In [269]:
storage_analysis = pd.DataFrame()
storage_analysis['Price'] = pc_price['Price']
storage_analysis['Storage Type'] = pc_price['Memory'].str.contains('SSD').map({True: 'Contains SSD', False: 'HDD / Other Only'})

fig = px.box(
    storage_analysis, 
    x='Storage Type', 
    y='Price', 
    color='Storage Type',
    title='Price Distribution: New Storage (SSD) vs. Traditional Storage (HDD)',
    labels={'Price': 'Laptop Price ($)'},
)
fig.show()

In [254]:
# Encode The Data
cols = ['Company', 'TypeName', 'Gpu', 'OpSys', 'Cpu']

features_df = pc_price[cols].copy()
features_df['WeightWithoutUnit'] = pc_price['Weight'].str.extract(r'(\d+\.?\d*)').astype(float)
features_df['RamWithoutUnit'] = pc_price['Ram'].str.extract(r'(\d+)').astype(int)
features_df['SSD'] = pc_price['Memory'].str.contains('SSD').astype(int)

df_resolution = pc_price['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
features_df['Resolution_Width'] = df_resolution[0].astype(int)
features_df['Resolution_Height'] = df_resolution[1].astype(int)

pc_price_encode = pd.get_dummies(features_df, dtype=int)
pc_price_encode['Price'] = pc_price['Price'].astype(int)

pc_price_encode['WeightWithoutUnit'] = pc_price_encode['WeightWithoutUnit'].fillna(pc_price_encode['WeightWithoutUnit'].median())
pc_price_encode['RamWithoutUnit'] = pc_price_encode['RamWithoutUnit'].fillna(pc_price_encode['RamWithoutUnit'].median())

# Split The Encoded Data
X = pc_price_encode.drop(columns='Price')
y = pc_price_encode['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=22)

# Scaling The Data
s = StandardScaler()
X_train_scaled = s.fit_transform(X_train)
X_test_scaled = s.transform(X_test)

# Check values std and mean
print("Mean:", np.mean(X_train_scaled))
print("Std Dev:", np.std(X_train_scaled))

Mean: 3.5438206947928136e-18
Std Dev: 0.9559590356822941


In [284]:
pc_price['Inches'] = pd.to_numeric(pc_price['Inches'], errors='coerce')
pc_price['Inches'] = pc_price['Inches'].fillna(pc_price['Inches'].median())
screen = pd.DataFrame()
screen['Inches'] = pc_price['Inches']
screen['Resolution_Width'] = pc_price_encode['Resolution_Width']
screen['Price'] = pc_price['Price']

fig = px.scatter(
    screen,
    x='Inches',
    y='Resolution_Width',
    color='Price',
    size='Price',
    title='Laptop Price Scaling by Screen Size and Resolution Width',
    labels={'Resolution_Width': 'Resolution Width (Pixels)', 'Inches': 'Screen Size (Inches)'},
    color_continuous_scale='Viridis'
)
fig.show()

In [255]:
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
ratios = pca.explained_variance_ratio_
print("Variance kept by the first 5 components:")
print(ratios[:5].round(3))

Variance kept by the first 5 components:
[0.02  0.018 0.014 0.013 0.012]


**Predict the price of a laptop (a continuous value) from its specifications: processor, RAM,
storage, GPU, screen size, and more.**

In [256]:
m = LinearRegression()
m.fit(X_train_pca, y_train)

predicted_value_lr = m.predict(X_test_pca)

In [264]:

rf_param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 0.5, None]
}

rf_grid_search = GridSearchCV(
    RandomForestRegressor(random_state=22),
    rf_param_grid, cv=5, scoring='r2', n_jobs=-1
)
rf_grid_search.fit(X_train_pca, y_train)
predicted_value_rf = rf_grid_search.best_estimator_.predict(X_test_pca)

In [258]:
dt_param_grid = {
    'max_depth': [3, 5, 10, 15, None],
    'min_samples_leaf': [2, 5, 10, 20, 50]
}

dt_grid_search = GridSearchCV(estimator=DecisionTreeRegressor(random_state=22), param_grid=dt_param_grid, cv=5, scoring='r2', n_jobs=-1)

dt_grid_search.fit(X_train_pca, y_train)
p_dt = dt_grid_search.predict(X_test_pca)

In [259]:
def score(n,y_test,predicted_value):
    print(f"MAE {n} {round(mean_absolute_error(y_test, predicted_value),3)}")
    print(f"MSE {n} {round(mean_squared_error(y_test, predicted_value),3)}")
    print(f"R2 {n} {round(r2_score(y_test, predicted_value),3)}")
    print()

In [265]:
score("Linear Regression Score: ",y_test, predicted_value_lr)

score("Random Forest Score: ",y_test, predicted_value_rf)

score("Decision Tree Score: ",y_test, p_dt)

MAE Linear Regression Score:  15003.034
MSE Linear Regression Score:  453992728.129
R2 Linear Regression Score:  0.683

MAE Random Forest Score:  11383.648
MSE Random Forest Score:  287166578.879
R2 Random Forest Score:  0.8

MAE Decision Tree Score:  13458.128
MSE Decision Tree Score:  389596933.476
R2 Decision Tree Score:  0.728



In [261]:
eval_df = pd.DataFrame({'Actual': y_test, 'Predicted': predicted_value_rf})
px.scatter(eval_df, x='Actual', y='Predicted', trendline="ols",  title="Actual vs Predicted Prices (Random Forest)").show()

# **In Conclusion:**
Explains how transforming 267 dummy features to 10 principal components lowered data dimensionality.
Model Comparison Scatter shows and evaluates how the Random Forest has the better linear regression results amoung LR and DT.  
### Insight 1
Discusses the direct linear correlation with Heatmap between CPU and GPU how they are simiiler in impcating prices.
### Insight 2
The graph shows that Razer is the company that sells the most expensive computers among them, unlike Vero, which has lower prices and is more suitable for low budget.
### Insight 3
Breaks down the price variance structural difference caused by modern component adoption (SSD vs. HDD)
### Insight 4
As we know the screens sit at a high price due to pixel density.